In [ ]:
# @title ⚡ Krea 2 满血生图指挥舱（Ultra A100/H100/G4 · L4/T4 全适配单 Cell）
# @markdown **点击一次 `▶` 一镜到底跑完**（支持 `Krea-2-Turbo` 8步极速 / `Krea-2-Raw` 原生底座 + 9套官方风格 LoRA 热切换 + 自动挂 Google Drive 防丢 + 常驻显存 Gradio 交互画室）：
# @markdown - **首次运行前提**：官方 `krea/Krea-2-Turbo` / `krea/Krea-2-Raw` 仓库需在 Hugging Face 页面点击 **Agree**（秒批），并填入 `HF_TOKEN`（推荐存进左侧钥匙图标 **Colab Secrets** 命名为 `HF_TOKEN`，或直接填在下方表单）。
# @markdown - **Ultra 20x 推荐选卡**：右上角切换 `A100` / `H100` / `G4 96GB` 时自动开启**原生 BF16 整模常驻显存**，1024×1024 Turbo 秒级出图，支持直出 1.5K–2K。

MODEL_VARIANT = "krea/Krea-2-Turbo"  # @param ["krea/Krea-2-Turbo", "krea/Krea-2-Raw"]
OFFICIAL_LORA = "none"  # @param ["none", "krea/Krea-2-LoRA-retroanime", "krea/Krea-2-LoRA-neondrip", "krea/Krea-2-LoRA-softwatercolor", "krea/Krea-2-LoRA-rainywindow", "krea/Krea-2-LoRA-vintagetarot", "krea/Krea-2-LoRA-darkbrush", "krea/Krea-2-LoRA-sunsetblur", "krea/Krea-2-LoRA-dotmatrix", "krea/Krea-2-LoRA-kidsdrawing"]
AUTO_APPEND_LORA_TRIGGER = True  # @param {type:"boolean"}
CUSTOM_LORA_REPO = ""  # @param {type:"string"}
LORA_SCALE = 1.0  # @param {type:"slider", min:0.0, max:1.5, step:0.05}
PROMPT = "A sleek futuristic neon cat director wearing glowing cyan visor glasses, directing a high-fashion cyberpunk cinema shoot in rainy Tokyo at night, wet asphalt reflections, anamorphic lens flare, 35mm Kodak film grain, ultra-detailed editorial photography"  # @param {type:"string"}
NEGATIVE_PROMPT = ""  # @param {type:"string"}
WIDTH = 1024  # @param {type:"integer"}
HEIGHT = 1024  # @param {type:"integer"}
STEPS = 0  # @param {type:"integer"}
GUIDANCE_SCALE = -1.0  # @param {type:"number"}
NUM_IMAGES = 2  # @param {type:"integer"}
SEED = -1  # @param {type:"integer"}
VRAM_MODE = "auto"  # @param ["auto", "full_gpu_bf16", "group_offload", "sequential_offload"]
HF_TOKEN = ""  # @param {type:"string"}
SAVE_TO_DRIVE = True  # @param {type:"boolean"}
LAUNCH_GRADIO_UI = True  # @param {type:"boolean"}

import getpass
import json
import os
import subprocess
import sys
import time
from pathlib import Path

os.environ["PIP_DISABLE_PIP_VERSION_CHECK"] = "1"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

print("=== [1/4] 硬件与运行环境自检 ===")
gpu_probe = subprocess.run(
    ["nvidia-smi", "--query-gpu=name,memory.total,driver_version", "--format=csv,noheader"],
    capture_output=True,
    text=True,
)
if gpu_probe.returncode != 0 or not gpu_probe.stdout.strip():
    raise RuntimeError("未检测到 NVIDIA GPU！请在 Colab 菜单栏『代码执行程序 -> 更改运行时类型』中选择 A100 / H100 / G4 / L4 / T4。")
print("GPU:", gpu_probe.stdout.strip())

# 挂载 Google Drive（云端防丢铁律）
run_stamp = time.strftime("%Y%m%d_%H%M%S")
out_dir = Path(f"/content/krea2_outputs/{run_stamp}")
if SAVE_TO_DRIVE:
    try:
        from google.colab import drive

        if not os.path.ismount("/content/drive"):
            print("正在挂载 Google Drive（成图与参数日志自动持久化防丢）...")
            drive.mount("/content/drive", force_remount=False)
        out_dir = Path(f"/content/drive/MyDrive/Krea2_Studio_Outputs/{run_stamp}")
    except Exception as drive_err:
        print(f"⚠️ Google Drive 挂载跳过（{drive_err}），改存本地目录: {out_dir}")
out_dir.mkdir(parents=True, exist_ok=True)
print("输出目录:", out_dir)

# 读取 HF_TOKEN（优先级：表单填写 > Colab Secrets > 环境变量 > 交互输入）
resolved_token = (HF_TOKEN or "").strip()
if not resolved_token:
    try:
        from google.colab import userdata

        for secret_name in ("HF_TOKEN", "HUGGINGFACE_HUB_TOKEN", "HF_API_TOKEN"):
            try:
                val = userdata.get(secret_name)
                if val and val.strip():
                    resolved_token = val.strip()
                    print(f"已从 Colab Secrets 自动读取 `{secret_name}`。")
                    break
            except Exception:
                pass
    except Exception:
        pass
if not resolved_token:
    resolved_token = (os.environ.get("HF_TOKEN") or os.environ.get("HUGGINGFACE_HUB_TOKEN") or "").strip()
if not resolved_token:
    print("\n🔑 未检测到 HF_TOKEN。Krea 2 官方权重为自动审批协议仓库：")
    print(f"   1. 请先访问 https://huggingface.co/{MODEL_VARIANT} 点击 Agree")
    print("   2. 在 https://huggingface.co/settings/tokens 获取 Read Token 后粘贴到下方（留空则尝试无 Token 访问）：")
    try:
        resolved_token = getpass.getpass("粘贴 Hugging Face Token (hf_...): ").strip()
    except Exception:
        resolved_token = ""

print("\n=== [2/4] 检查并安装 Krea 2 依赖（单 Cell 零重启） ===")
check_cmd = [
    sys.executable,
    "-c",
    "from diffusers import Krea2Pipeline; from transformers import Qwen3VLModel; import accelerate, peft, safetensors, huggingface_hub, gradio; print('DEPS_OK')",
]
dep_check = subprocess.run(check_cmd, capture_output=True, text=True)
if dep_check.returncode != 0 or "DEPS_OK" not in dep_check.stdout:
    print("正在安装/升级 diffusers (含 Krea2Pipeline) + transformers (含 Qwen3-VL) + peft + gradio ...")
    base_pkgs = [
        "transformers>=4.51.0",
        "accelerate>=1.3.0",
        "safetensors>=0.4.5",
        "peft>=0.14.0",
        "sentencepiece",
        "huggingface_hub>=0.28.0",
        "gradio>=5.0.0",
        "pillow",
        "diffusers>=0.43.0",
    ]
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U", *base_pkgs], check=True)
    recheck = subprocess.run(
        [sys.executable, "-c", "from diffusers import Krea2Pipeline; print('OK')"],
        capture_output=True,
        text=True,
    )
    if recheck.returncode != 0 or "OK" not in recheck.stdout:
        print("PyPI 版 diffusers 暂未包含 Krea2Pipeline，正在拉取 Hugging Face diffusers 最新主干...")
        subprocess.run(
            [sys.executable, "-m", "pip", "install", "-q", "-U", "git+https://github.com/huggingface/diffusers.git"],
            check=True,
        )
    subprocess.run(check_cmd, check=True)
else:
    print("依赖已就绪，跳过安装。")

print("\n=== [3/4] 写入独立推理引擎脚本 ===")
worker_path = Path("/content/krea2_studio_worker.py")
config_path = Path("/content/krea2_run_config.json")
worker_code = '#!/usr/bin/env python3\n"""Krea 2 Studio Worker — Single-Cell Colab Backend (A100/H100/G4/L4/T4 Adaptive)."""\n\nimport gc\nimport json\nimport os\nimport random\nimport sys\nimport time\nimport traceback\nfrom pathlib import Path\n\nimport torch\nfrom PIL import Image\n\nOFFICIAL_LORAS = {\n    "krea/Krea-2-LoRA-retroanime": ("retroanime.safetensors", "Purple retro anime style"),\n    "krea/Krea-2-LoRA-neondrip": ("neondrip.safetensors", "Textured abstract style"),\n    "krea/Krea-2-LoRA-softwatercolor": ("softwatercolor.safetensors", "Art Deco watercolor style"),\n    "krea/Krea-2-LoRA-rainywindow": ("rainywindow.safetensors", "rainy window style"),\n    "krea/Krea-2-LoRA-vintagetarot": ("vintagetarot.safetensors", "vintage tarot style"),\n    "krea/Krea-2-LoRA-darkbrush": ("darkbrush.safetensors", "monochrome ink wash style"),\n    "krea/Krea-2-LoRA-sunsetblur": ("sunsetblur.safetensors", "ethereal motion blur style"),\n    "krea/Krea-2-LoRA-dotmatrix": ("dotmatrix.safetensors", "Monochrome stippling style"),\n    "krea/Krea-2-LoRA-kidsdrawing": ("kidsdrawing.safetensors", "naive expressive sketch style"),\n}\n\nASPECT_PRESETS = {\n    "1:1 标准方图 (1024×1024)": (1024, 1024),\n    "16:9 电影横屏 (1344×768)": (1344, 768),\n    "9:16 竖屏壁纸/海报 (768×1344)": (768, 1344),\n    "3:2 摄影横构图 (1248×832)": (1248, 832),\n    "2:3 肖像竖构图 (832×1248)": (832, 1248),\n    "4:3 复古画幅 (1152×896)": (1152, 896),\n    "1:1 高清 1.5K (1536×1536, 推荐A100/H100/G4)": (1536, 1536),\n    "16:9 高清 2K 电影宽屏 (1920×1088, 推荐A100/H100/G4)": (1920, 1088),\n}\n\n\ndef snap16(val: int, lo: int = 512, hi: int = 2048) -> int:\n    val = max(lo, min(hi, int(val)))\n    return max(lo, (val // 16) * 16)\n\n\ndef resolve_defaults(model_id: str, steps: int, guidance_scale: float):\n    is_turbo = "turbo" in model_id.lower()\n    final_steps = int(steps) if int(steps) > 0 else (8 if is_turbo else 32)\n    final_cfg = float(guidance_scale) if float(guidance_scale) >= 0.0 else (0.0 if is_turbo else 4.0)\n    return final_steps, final_cfg\n\n\ndef detect_gpu():\n    if not torch.cuda.is_available():\n        raise RuntimeError("未检测到 CUDA GPU！请在 Colab 菜单栏『代码执行程序 -> 更改运行时类型』中选择 A100 / H100 / G4 / L4 / T4 GPU。")\n    props = torch.cuda.get_device_properties(0)\n    vram_gb = props.total_memory / (1024 ** 3)\n    return props.name, vram_gb\n\n\ndef choose_vram_mode(requested: str, vram_gb: float) -> str:\n    if requested != "auto":\n        return requested\n    if vram_gb >= 38.0:\n        return "full_gpu_bf16"\n    if vram_gb >= 20.0:\n        return "group_offload"\n    return "sequential_offload"\n\n\nclass Krea2Engine:\n    def __init__(self, model_id: str, vram_mode: str, hf_token: str | None, out_dir: Path):\n        self.model_id = model_id\n        self.vram_mode = vram_mode\n        self.hf_token = hf_token or None\n        self.out_dir = out_dir\n        self.out_dir.mkdir(parents=True, exist_ok=True)\n        self.active_lora_key = None\n        self.active_lora_scale = None\n        self.pipe = None\n        self.gpu_name, self.vram_gb = detect_gpu()\n        self.resolved_vram_mode = choose_vram_mode(vram_mode, self.vram_gb)\n        self._load_pipeline()\n\n    def _load_pipeline(self):\n        from diffusers import Krea2Pipeline\n\n        torch.backends.cuda.matmul.allow_tf32 = True\n        if hasattr(torch, "set_float32_matmul_precision"):\n            torch.set_float32_matmul_precision("high")\n\n        print(f"[Krea2] 检测到 GPU: {self.gpu_name} ({self.vram_gb:.1f} GB VRAM)")\n        print(f"[Krea2] 显存策略: {self.resolved_vram_mode} | 正在加载模型: {self.model_id} ...")\n        t0 = time.time()\n\n        try:\n            pipe = Krea2Pipeline.from_pretrained(\n                self.model_id,\n                torch_dtype=torch.bfloat16,\n                token=self.hf_token,\n            )\n        except Exception as e:\n            msg = str(e)\n            if any(k in msg for k in ("401", "403", "GatedRepo", "gated", "Unauthorized", "Access to model")):\n                print("\\n" + "=" * 76)\n                print("❌ Hugging Face 权限校验未通过（Krea 2 官方仓库需要一键同意开源协议 + HF_TOKEN）")\n                print("👉 第 1 步：打开下面链接，登录 Hugging Face 并点击 Agree（秒批）：")\n                print(f"   https://huggingface.co/{self.model_id}")\n                print("👉 第 2 步：在这里创建 Read 权限 Token：")\n                print("   https://huggingface.co/settings/tokens")\n                print("👉 第 3 步：将 Token 填入 Colab 左侧钥匙图标 Secrets（名称 HF_TOKEN）或本单元格表单 HF_TOKEN 后重跑。")\n                print("=" * 76 + "\\n")\n            raise\n\n        if self.resolved_vram_mode == "full_gpu_bf16":\n            pipe = pipe.to("cuda")\n        elif self.resolved_vram_mode == "group_offload":\n            try:\n                pipe.enable_group_offload(\n                    onload_device=torch.device("cuda"),\n                    offload_device=torch.device("cpu"),\n                    offload_type="block_level",\n                    num_blocks_per_group=2,\n                    use_stream=True,\n                )\n            except Exception as offload_err:\n                print(f"[Krea2] group_offload 不可用 ({offload_err})，自动回退为 sequential_cpu_offload ...")\n                pipe.enable_sequential_cpu_offload()\n        else:\n            pipe.enable_sequential_cpu_offload()\n\n        self.pipe = pipe\n        print(f"[Krea2] 模型加载完成！耗时 {time.time() - t0:.1f}s")\n\n    def _clear_lora(self):\n        if not self.active_lora_key:\n            return\n        try:\n            if hasattr(self.pipe.transformer, "delete_adapters"):\n                self.pipe.transformer.delete_adapters(["krea_style"])\n            elif hasattr(self.pipe, "unload_lora_weights"):\n                self.pipe.unload_lora_weights()\n        except Exception:\n            try:\n                self.pipe.unload_lora_weights()\n            except Exception:\n                pass\n        self.active_lora_key = None\n        self.active_lora_scale = None\n        gc.collect()\n        torch.cuda.empty_cache()\n\n    def _resolve_lora_weight_name(self, lora_repo: str) -> str | None:\n        if lora_repo in OFFICIAL_LORAS:\n            return OFFICIAL_LORAS[lora_repo][0]\n        if os.path.isfile(lora_repo):\n            return None\n        try:\n            from huggingface_hub import HfApi\n\n            files = HfApi(token=self.hf_token).list_repo_files(lora_repo)\n            safes = [f for f in files if f.endswith(".safetensors")]\n            if not safes:\n                return None\n            for preferred in ("pytorch_lora_weights.safetensors",):\n                if preferred in safes:\n                    return preferred\n            return safes[0]\n        except Exception as e:\n            print(f"[Krea2] 自动检索 LoRA 权重文件失败 ({e})，将尝试默认加载。")\n            return None\n\n    def set_lora(self, official_lora: str, custom_lora: str, scale: float):\n        target = (custom_lora or "").strip()\n        if not target and official_lora and official_lora != "none":\n            target = official_lora.strip()\n\n        if not target or float(scale) <= 0.0:\n            self._clear_lora()\n            return None\n\n        if self.active_lora_key == target:\n            if self.active_lora_scale != float(scale):\n                try:\n                    self.pipe.transformer.set_adapters("krea_style", weights=float(scale))\n                except Exception:\n                    try:\n                        self.pipe.set_adapters(["krea_style"], adapter_weights=[float(scale)])\n                    except Exception:\n                        pass\n                self.active_lora_scale = float(scale)\n            return target\n\n        self._clear_lora()\n        print(f"[Krea2] 正在加载 LoRA: {target} (weight={scale:.2f}) ...")\n\n        if os.path.isfile(target):\n            lora_path = Path(target)\n            repo_or_dir = str(lora_path.parent)\n            weight_name = lora_path.name\n        else:\n            repo_or_dir = target\n            weight_name = self._resolve_lora_weight_name(target)\n\n        load_kwargs = {"adapter_name": "krea_style"}\n        if weight_name:\n            load_kwargs["weight_name"] = weight_name\n        if self.hf_token and not os.path.isdir(repo_or_dir):\n            load_kwargs["token"] = self.hf_token\n\n        try:\n            self.pipe.transformer.load_lora_adapter(repo_or_dir, **load_kwargs)\n            self.pipe.transformer.set_adapters("krea_style", weights=float(scale))\n        except Exception as e1:\n            print(f"[Krea2] transformer.load_lora_adapter 兼容回退 ({e1})，改用 pipe.load_lora_weights ...")\n            self.pipe.load_lora_weights(repo_or_dir, **load_kwargs)\n            try:\n                self.pipe.set_adapters(["krea_style"], adapter_weights=[float(scale)])\n            except Exception:\n                pass\n\n        self.active_lora_key = target\n        self.active_lora_scale = float(scale)\n        print(f"[Krea2] LoRA 已生效: {target}")\n        return target\n\n    def prepare_prompt(self, prompt: str, official_lora: str, custom_lora: str, auto_trigger: bool) -> str:\n        clean = (prompt or "").strip()\n        if not clean:\n            clean = "a sleek neon cat director filming a cinematic scene in rainy Tokyo, 35mm film"\n        if auto_trigger and not (custom_lora or "").strip() and official_lora in OFFICIAL_LORAS:\n            trigger = OFFICIAL_LORAS[official_lora][1]\n            if trigger.lower() not in clean.lower():\n                clean = f"{clean}, {trigger}"\n        return clean\n\n    def generate(\n        self,\n        prompt: str,\n        negative_prompt: str = "",\n        width: int = 1024,\n        height: int = 1024,\n        steps: int = 0,\n        guidance_scale: float = -1.0,\n        num_images: int = 1,\n        seed: int = -1,\n        official_lora: str = "none",\n        custom_lora: str = "",\n        lora_scale: float = 1.0,\n        auto_append_trigger: bool = True,\n    ):\n        active_lora = self.set_lora(official_lora, custom_lora, lora_scale)\n        final_prompt = self.prepare_prompt(prompt, official_lora, custom_lora, auto_append_trigger)\n        w = snap16(width)\n        h = snap16(height)\n        final_steps, final_cfg = resolve_defaults(self.model_id, steps, guidance_scale)\n        n = max(1, min(8, int(num_images)))\n        base_seed = int(seed) if int(seed) >= 0 else random.randint(0, 2 ** 31 - 1)\n\n        saved_paths = []\n        pil_images = []\n        records = []\n\n        gen_device = "cuda" if self.resolved_vram_mode == "full_gpu_bf16" else "cpu"\n\n        for i in range(n):\n            cur_seed = base_seed + i\n            generator = torch.Generator(device=gen_device).manual_seed(cur_seed)\n            t0 = time.time()\n\n            call_kwargs = {\n                "prompt": final_prompt,\n                "width": w,\n                "height": h,\n                "num_inference_steps": final_steps,\n                "guidance_scale": final_cfg,\n                "generator": generator,\n            }\n            if final_cfg > 0.0 and (negative_prompt or "").strip():\n                call_kwargs["negative_prompt"] = negative_prompt.strip()\n\n            with torch.inference_mode():\n                out = self.pipe(**call_kwargs)\n            img = out.images[0]\n            elapsed = time.time() - t0\n\n            ts = time.strftime("%Y%m%d_%H%M%S")\n            fname = f"krea2_{ts}_s{cur_seed}_{w}x{h}_{i + 1:02d}.png"\n            fpath = self.out_dir / fname\n            img.save(fpath)\n\n            rec = {\n                "file": str(fpath),\n                "filename": fname,\n                "model": self.model_id,\n                "lora": active_lora,\n                "lora_scale": float(lora_scale) if active_lora else 0.0,\n                "prompt": final_prompt,\n                "negative_prompt": negative_prompt if final_cfg > 0 else "",\n                "width": w,\n                "height": h,\n                "steps": final_steps,\n                "guidance_scale": final_cfg,\n                "seed": cur_seed,\n                "elapsed_sec": round(elapsed, 2),\n            }\n            records.append(rec)\n            saved_paths.append(str(fpath))\n            pil_images.append(img)\n            print(f"[Krea2] ({i + 1}/{n}) 出图完成 {w}×{h} | steps={final_steps} cfg={final_cfg} seed={cur_seed} | {elapsed:.2f}s -> {fpath}")\n\n        manifest_path = self.out_dir / "manifest.jsonl"\n        with manifest_path.open("a", encoding="utf-8") as mf:\n            for r in records:\n                mf.write(json.dumps(r, ensure_ascii=False) + "\\n")\n\n        Path("/content/krea2_last_batch.json").write_text(\n            json.dumps({"out_dir": str(self.out_dir), "records": records}, ensure_ascii=False, indent=2),\n            encoding="utf-8",\n        )\n        return pil_images, saved_paths, records\n\n\ndef launch_gradio(engine: Krea2Engine, cfg: dict, initial_paths: list[str], initial_summary: str):\n    import gradio as gr\n\n    lora_choices = ["none"] + list(OFFICIAL_LORAS.keys())\n    init_steps, init_cfg = resolve_defaults(cfg["MODEL_VARIANT"], cfg["STEPS"], cfg["GUIDANCE_SCALE"])\n\n    def apply_preset(preset_label: str):\n        w, h = ASPECT_PRESETS.get(preset_label, (1024, 1024))\n        return w, h\n\n    def on_generate(\n        prompt,\n        negative_prompt,\n        width,\n        height,\n        steps,\n        guidance_scale,\n        num_images,\n        seed,\n        official_lora,\n        custom_lora,\n        lora_scale,\n        auto_trigger,\n    ):\n        try:\n            imgs, paths, recs = engine.generate(\n                prompt=prompt,\n                negative_prompt=negative_prompt,\n                width=int(width),\n                height=int(height),\n                steps=int(steps),\n                guidance_scale=float(guidance_scale),\n                num_images=int(num_images),\n                seed=int(seed),\n                official_lora=official_lora,\n                custom_lora=custom_lora,\n                lora_scale=float(lora_scale),\n                auto_append_trigger=bool(auto_trigger),\n            )\n            lines = [\n                f"✅ 完成 {len(recs)} 张 | 模型 `{engine.model_id}` | 显存 `{engine.gpu_name} ({engine.vram_gb:.1f}GB)` | 输出目录: `{engine.out_dir}`"\n            ]\n            for r in recs:\n                lines.append(\n                    f"- `{r[\'filename\']}` | {r[\'width\']}×{r[\'height\']} | steps={r[\'steps\']} | cfg={r[\'guidance_scale\']} | seed={r[\'seed\']} | {r[\'elapsed_sec\']}s"\n                )\n            return paths, "\\n".join(lines)\n        except Exception as exc:\n            tb = traceback.format_exc()\n            return [], f"❌ 生成失败: {exc}\\n\\n```\\n{tb[-1500:]}\\n```"\n\n    def on_batch_generate(\n        prompts_multiline,\n        negative_prompt,\n        width,\n        height,\n        steps,\n        guidance_scale,\n        seed,\n        official_lora,\n        custom_lora,\n        lora_scale,\n        auto_trigger,\n    ):\n        prompt_list = [ln.strip() for ln in (prompts_multiline or "").splitlines() if ln.strip()]\n        if not prompt_list:\n            return [], "⚠️ 请至少输入一行 Prompt（每行一条提示词）。"\n        all_paths = []\n        all_lines = [f"✅ 批量连抽完成（共 {len(prompt_list)} 条 Prompt）| 输出目录: `{engine.out_dir}`"]\n        base_seed = int(seed) if int(seed) >= 0 else random.randint(0, 2 ** 31 - 1)\n        try:\n            for idx, p in enumerate(prompt_list):\n                _, paths, recs = engine.generate(\n                    prompt=p,\n                    negative_prompt=negative_prompt,\n                    width=int(width),\n                    height=int(height),\n                    steps=int(steps),\n                    guidance_scale=float(guidance_scale),\n                    num_images=1,\n                    seed=base_seed + idx,\n                    official_lora=official_lora,\n                    custom_lora=custom_lora,\n                    lora_scale=float(lora_scale),\n                    auto_append_trigger=bool(auto_trigger),\n                )\n                all_paths.extend(paths)\n                r = recs[0]\n                all_lines.append(f"- #{idx + 1} `{r[\'filename\']}` | seed={r[\'seed\']} | {r[\'elapsed_sec\']}s")\n            return all_paths, "\\n".join(all_lines)\n        except Exception as exc:\n            tb = traceback.format_exc()\n            return all_paths, f"❌ 批量生成中途出错: {exc}\\n\\n```\\n{tb[-1500:]}\\n```"\n\n    with gr.Blocks(title="Krea 2 Ultra Studio") as demo:\n        gr.Markdown(\n            f"# ⚡ Krea 2 满血生图工作台（Colab 单 Cell）\\n"\n            f"**当前硬件**: `{engine.gpu_name}` (`{engine.vram_gb:.1f} GB VRAM`) · **精度模式**: `{engine.resolved_vram_mode}` · **当前底座**: `{engine.model_id}` · **自动归档**: `{engine.out_dir}`"\n        )\n        with gr.Tabs():\n            with gr.Tab("🎨 单条精调 / 抽卡"):\n                with gr.Row():\n                    with gr.Column(scale=5):\n                        prompt_in = gr.Textbox(label="Prompt（提示词，推荐英文精细描写光影、构图与质感）", value=cfg["PROMPT"], lines=4)\n                        neg_in = gr.Textbox(label="Negative Prompt（仅 guidance_scale > 0 如 Krea-2-Raw 时生效）", value=cfg["NEGATIVE_PROMPT"], lines=2)\n                        preset_dd = gr.Dropdown(\n                            label="快速画幅预设",\n                            choices=list(ASPECT_PRESETS.keys()),\n                            value="1:1 标准方图 (1024×1024)",\n                        )\n                        with gr.Row():\n                            w_in = gr.Slider(512, 2048, value=snap16(cfg["WIDTH"]), step=16, label="宽度 Width")\n                            h_in = gr.Slider(512, 2048, value=snap16(cfg["HEIGHT"]), step=16, label="高度 Height")\n                        with gr.Row():\n                            steps_in = gr.Slider(4, 60, value=init_steps, step=1, label="步数 Steps（Turbo=8 / Raw=28~52）")\n                            cfg_in = gr.Slider(0.0, 8.0, value=init_cfg, step=0.1, label="CFG（Turbo=0.0 / Raw=3.5~4.5）")\n                        with gr.Row():\n                            n_in = gr.Slider(1, 4, value=min(4, max(1, int(cfg["NUM_IMAGES"]))), step=1, label="单次张数")\n                            seed_in = gr.Number(value=int(cfg["SEED"]), precision=0, label="Seed（-1 随机）")\n                        with gr.Accordion("🎨 官方 / 自定义风格 LoRA（支持免重启热切换）", open=True):\n                            lora_dd = gr.Dropdown(label="官方 Krea 2 LoRA", choices=lora_choices, value=cfg["OFFICIAL_LORA"])\n                            custom_lora_in = gr.Textbox(label="自定义 LoRA（HF Repo 或 Drive `.safetensors` 路径，留空用上方官方预设）", value=cfg["CUSTOM_LORA_REPO"])\n                            with gr.Row():\n                                lora_scale_in = gr.Slider(0.0, 1.5, value=float(cfg["LORA_SCALE"]), step=0.05, label="LoRA 强度")\n                                auto_trig_in = gr.Checkbox(value=bool(cfg["AUTO_APPEND_LORA_TRIGGER"]), label="自动补全官方 LoRA 触发词")\n                        run_btn = gr.Button("🚀 立即生图（模型常驻显存，秒级出图）", variant="primary")\n                    with gr.Column(scale=6):\n                        gallery_out = gr.Gallery(label="成图画廊（点击可放大/下载，同时已自动备份）", value=initial_paths, columns=2, height="auto")\n                        status_md = gr.Markdown(value=initial_summary)\n\n            with gr.Tab("📋 批量分镜 / 多 Prompt 连抽"):\n                with gr.Row():\n                    with gr.Column(scale=5):\n                        batch_prompts_in = gr.Textbox(\n                            label="批量提示词队列（每行一条 Prompt，适合一键跑多镜头/多构图方案）",\n                            placeholder="每行输入一条 Prompt...\\n例如：\\nLine 1: Wide establishing shot...\\nLine 2: Close-up portrait...",\n                            lines=8,\n                        )\n                        batch_run_btn = gr.Button("⚡ 按当前画幅/LoRA参数批量连抽", variant="primary")\n                    with gr.Column(scale=6):\n                        batch_gallery_out = gr.Gallery(label="批量成图画廊", columns=2, height="auto")\n                        batch_status_md = gr.Markdown(value="粘贴多行 Prompt 后点击按钮即可按顺序连续出图并自动归档至 Google Drive。")\n\n        preset_dd.change(apply_preset, inputs=[preset_dd], outputs=[w_in, h_in])\n        run_btn.click(\n            on_generate,\n            inputs=[\n                prompt_in,\n                neg_in,\n                w_in,\n                h_in,\n                steps_in,\n                cfg_in,\n                n_in,\n                seed_in,\n                lora_dd,\n                custom_lora_in,\n                lora_scale_in,\n                auto_trig_in,\n            ],\n            outputs=[gallery_out, status_md],\n        )\n        batch_run_btn.click(\n            on_batch_generate,\n            inputs=[\n                batch_prompts_in,\n                neg_in,\n                w_in,\n                h_in,\n                steps_in,\n                cfg_in,\n                seed_in,\n                lora_dd,\n                custom_lora_in,\n                lora_scale_in,\n                auto_trig_in,\n            ],\n            outputs=[batch_gallery_out, batch_status_md],\n        )\n\n    print("\\n[Krea2] 正在启动 Gradio 交互式画室（模型常驻显存，无需重载）...")\n    demo.queue().launch(share=True, debug=False)\n\n\ndef main():\n    if len(sys.argv) < 2:\n        raise SystemExit("Usage: krea2_studio_worker.py <config.json>")\n    cfg = json.loads(Path(sys.argv[1]).read_text(encoding="utf-8"))\n    hf_token = os.environ.get("HF_TOKEN") or os.environ.get("HUGGINGFACE_HUB_TOKEN") or ""\n    out_dir = Path(cfg["OUT_DIR"])\n\n    engine = Krea2Engine(\n        model_id=cfg["MODEL_VARIANT"],\n        vram_mode=cfg["VRAM_MODE"],\n        hf_token=hf_token,\n        out_dir=out_dir,\n    )\n\n    _, initial_paths, initial_records = engine.generate(\n        prompt=cfg["PROMPT"],\n        negative_prompt=cfg["NEGATIVE_PROMPT"],\n        width=cfg["WIDTH"],\n        height=cfg["HEIGHT"],\n        steps=cfg["STEPS"],\n        guidance_scale=cfg["GUIDANCE_SCALE"],\n        num_images=cfg["NUM_IMAGES"],\n        seed=cfg["SEED"],\n        official_lora=cfg["OFFICIAL_LORA"],\n        custom_lora=cfg["CUSTOM_LORA_REPO"],\n        lora_scale=cfg["LORA_SCALE"],\n        auto_append_trigger=cfg["AUTO_APPEND_LORA_TRIGGER"],\n    )\n\n    lines = [\n        f"✅ 首轮出图完成（共 {len(initial_records)} 张）| 模型 `{engine.model_id}` | 已保存至 `{out_dir}`"\n    ]\n    for r in initial_records:\n        lines.append(\n            f"- `{r[\'filename\']}` | {r[\'width\']}×{r[\'height\']} | steps={r[\'steps\']} | cfg={r[\'guidance_scale\']} | seed={r[\'seed\']} | {r[\'elapsed_sec\']}s"\n        )\n    summary = "\\n".join(lines)\n    print("\\n" + summary)\n\n    if cfg.get("LAUNCH_GRADIO_UI", True):\n        launch_gradio(engine, cfg, initial_paths, summary)\n\n\nif __name__ == "__main__":\n    main()\n'
worker_path.write_text(worker_code, encoding="utf-8")

run_cfg = {
    "MODEL_VARIANT": MODEL_VARIANT,
    "OFFICIAL_LORA": OFFICIAL_LORA,
    "AUTO_APPEND_LORA_TRIGGER": bool(AUTO_APPEND_LORA_TRIGGER),
    "CUSTOM_LORA_REPO": CUSTOM_LORA_REPO,
    "LORA_SCALE": float(LORA_SCALE),
    "PROMPT": PROMPT,
    "NEGATIVE_PROMPT": NEGATIVE_PROMPT,
    "WIDTH": int(WIDTH),
    "HEIGHT": int(HEIGHT),
    "STEPS": int(STEPS),
    "GUIDANCE_SCALE": float(GUIDANCE_SCALE),
    "NUM_IMAGES": int(NUM_IMAGES),
    "SEED": int(SEED),
    "VRAM_MODE": VRAM_MODE,
    "OUT_DIR": str(out_dir),
    "LAUNCH_GRADIO_UI": bool(LAUNCH_GRADIO_UI),
}
config_path.write_text(json.dumps(run_cfg, ensure_ascii=False, indent=2), encoding="utf-8")

print("\n=== [4/4] 在全新子进程中拉起 Krea 2 生图引擎 ===")
env = os.environ.copy()
if resolved_token:
    env["HF_TOKEN"] = resolved_token
    env["HUGGINGFACE_HUB_TOKEN"] = resolved_token

proc = subprocess.run([sys.executable, "-u", str(worker_path), str(config_path)], env=env)
if proc.returncode != 0:
    raise RuntimeError(f"Krea 2 推理子进程异常退出 (exit_code={proc.returncode})，请检查上方日志。")

# 若未开启常驻 Gradio UI，则在 Colab 单元格下方直接内联展示成图
if not LAUNCH_GRADIO_UI:
    batch_file = Path("/content/krea2_last_batch.json")
    if batch_file.exists():
        from IPython.display import Image as IPImage, display

        info = json.loads(batch_file.read_text(encoding="utf-8"))
        for rec in info.get("records", []):
            print(f"🖼️ {rec['filename']} | seed={rec['seed']} | {rec['elapsed_sec']}s")
            display(IPImage(filename=rec["file"]))